In [1]:
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

# Transformer Decoder

The transformer decoder for lm is a stack of causal self-attention and feed-forward layers.
The positional encoding is a learnable parameter and it is added to the input of the decoder.

In [2]:
import torch

In [3]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [4]:
class FeedForward(torch.nn.Module):
    def __init__(self, d_model=512, d_ff=1024, dropout=0.1, **kwargs):
        super().__init__()
        self.ff = torch.nn.Sequential(
            torch.nn.LayerNorm(d_model),
            torch.nn.Linear(d_model, d_ff),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(d_ff, d_model),
        )        
    def forward(self, x):
        return self.ff(x)

class CausalSelfAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=8, d_head=64, dropout=0.1, seq_len=1024,**kwargs):
        super().__init__()
        print('Causal Self Attention, d_model:', d_model, 'n_heads:', n_heads, 'd_head:', d_head, 'seq_len:', seq_len)
        self.seq_len = seq_len
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_head
        self.scale = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))
        self.norm = torch.nn.LayerNorm(d_model)
        self.q_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.v_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.k_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.dropout = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(d_head*n_heads, d_model)        
        self.register_buffer("mask", torch.tril(torch.ones(self.seq_len, self.seq_len))[None, None, ...] == 0)
            
    def forward(self, x):
        x = self.norm(x)
        b, n, d = x.shape
        q = self.q_linear(x).reshape(b, -1, self.n_heads, self.d_head)
        k = self.k_linear(x).reshape(b, -1, self.n_heads, self.d_head)
        v = self.v_linear(x).reshape(b, -1, self.n_heads, self.d_head) 
        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale        
        scores = scores.masked_fill(self.mask[:,:,:n,:n], float('-inf'))
        att = scores.softmax(dim=-1)
        att = self.dropout(att)
        out = torch.einsum('bhij,bjhd->bihd', att, v)
        out = self.dropout(out).reshape(b, -1, self.n_heads*self.d_head)
        out = self.out(out)
        return out

class Decoder(torch.nn.Module):
    def __init__(self, nb_layers=6, **kwargs):
        super().__init__()        
        seq_len = kwargs['seq_len']     
        self.pos = torch.nn.Parameter(torch.randn(1, seq_len, kwargs['d_model']))
        self.att = torch.nn.ModuleList([CausalSelfAttention(**kwargs) for _ in range(nb_layers)])
        self.ff = torch.nn.ModuleList([FeedForward(**kwargs) for _ in range(nb_layers)])
        
    def forward(self, x):
        b, n, d = x.shape
        x = x + self.pos[:, :n, :]
        for att, ff in zip(self.att, self.ff):
            x = x + att(x)
            x = x + ff(x)            
        return x

class Transformer(torch.nn.Module):
    def __init__(self, vocab_size=25, eos_token_id=24, **kwargs):
        super().__init__()
        self.vocab_size = vocab_size
        self.eos_token_id = eos_token_id
        self.seq_len = kwargs['seq_len']    
        self.emb = torch.nn.Embedding(vocab_size, kwargs['d_model'])        
        self.dec = Decoder(**kwargs)
        self.out = torch.nn.Linear(kwargs['d_model'], vocab_size)
        
    def decoder(self, x):
        x = self.emb(x)
        x = self.dec(x)
        return self.out(x)
    
    def forward(self, x):
        return self.decoder(x)
                
    def loss(self, y):         
        logits = self(y[:,:-1]).reshape(-1, self.vocab_size)
        target = y[:,1:].reshape(-1)
        
        loss = torch.nn.functional.cross_entropy(logits,target)
        return loss
    
    def generate(self, y):
        device = next(self.parameters()).device
        self.eval()        
        y = y.tolist()
        
        with torch.no_grad():                        
            while y[-1] != self.eos_token_id and len(y) < self.seq_len:
                logits = self.decoder(torch.tensor(y).reshape(1,-1).to(device))
                y.append(logits.argmax(-1)[:,-1].item())
                
        return y

test a small model and loss with artifical data

In [5]:
model = Transformer(vocab_size=24, nb_layers=4, d_model=512, n_heads=2, d_head=32, dropout=0.1, seq_len=32)
x = torch.randint(0, 24, (1, 32))
print( model(x).shape )
print( model.loss(x) )

Causal Self Attention, d_model: 512 n_heads: 2 d_head: 32 seq_len: 32
Causal Self Attention, d_model: 512 n_heads: 2 d_head: 32 seq_len: 32
Causal Self Attention, d_model: 512 n_heads: 2 d_head: 32 seq_len: 32
Causal Self Attention, d_model: 512 n_heads: 2 d_head: 32 seq_len: 32


torch.Size([1, 32, 24])
tensor(3.4083, grad_fn=<NllLossBackward0>)


## Análisis del vocabulario de fechas2

Primero extraemos todas las palabras únicas del dataset para construir el tokenizador apropiado.

In [6]:
# Extraer vocabulario único de los archivos de datos
vocab = set()

# Leer archivo de entrenamiento
with open('../fechas2/fechas2_train.es.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()[1:]  # Saltar el encabezado
    for line in lines:
        parts = line.strip().split(',', 1)
        if len(parts) == 2:
            text = parts[1]
            vocab.update(text.split())

# Leer archivo de test
with open('../fechas2/fechas2_test.es.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()[1:]  # Saltar el encabezado
    for line in lines:
        parts = line.strip().split(',', 1)
        if len(parts) == 2:
            text = parts[1]
            vocab.update(text.split())

# Ordenar vocabulario
vocab_sorted = sorted(vocab)

print(f"Total de palabras únicas: {len(vocab_sorted)}")
print(f"\nVocabulario completo:")
for i, word in enumerate(vocab_sorted):
    print(f"  {i}: '{word}'")

Total de palabras únicas: 22

Vocabulario completo:
  0: 'de'
  1: 'días'
  2: 'el'
  3: 'en'
  4: 'este'
  5: 'favor'
  6: 'gracias'
  7: 'jueves'
  8: 'lunes'
  9: 'martes'
  10: 'mañana'
  11: 'miércoles'
  12: 'par'
  13: 'pasado'
  14: 'por'
  15: 'próximo'
  16: 'que'
  17: 'siguiente'
  18: 'tres'
  19: 'un'
  20: 'viene'
  21: 'viernes'


### Análisis de secuencias

Verificar longitud máxima de las secuencias para configurar seq_len apropiadamente.

In [7]:
# Analizar longitud de secuencias
max_len = 0
min_len = float('inf')
lengths = []

for file in ['../fechas2/fechas2_train.es.csv', '../fechas2/fechas2_test.es.csv']:
    with open(file, 'r', encoding='utf-8') as f:
        lines = f.readlines()[1:]
        for line in lines:
            parts = line.strip().split(',', 1)
            if len(parts) == 2:
                words = parts[1].split()
                seq_len = len(words)
                lengths.append(seq_len)
                max_len = max(max_len, seq_len)
                min_len = min(min_len, seq_len)

print(f"Longitud mínima de secuencia: {min_len} palabras")
print(f"Longitud máxima de secuencia: {max_len} palabras")
print(f"Longitud promedio: {sum(lengths)/len(lengths):.2f} palabras")
print(f"\nCon tokens especiales (<sos>, <eos>) la longitud máxima será: {max_len + 2}")
print(f"Recomendación: usar seq_len={max_len + 4} (deja margen para padding)")

Longitud mínima de secuencia: 1 palabras
Longitud máxima de secuencia: 7 palabras
Longitud promedio: 3.61 palabras

Con tokens especiales (<sos>, <eos>) la longitud máxima será: 9
Recomendación: usar seq_len=11 (deja margen para padding)


### Muestras de los datos

Veamos algunos ejemplos del dataset para entender su formato.

In [8]:
# Ejemplos del dataset
with open('../fechas2/fechas2_train.es.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()[1:11]  # Primeras 10 líneas
    for i, line in enumerate(lines, 1):
        parts = line.strip().split(',', 1)
        if len(parts) == 2:
            wav_path, text = parts
            print(f"{i}. Archivo: {wav_path}")
            print(f"   Texto: '{text}'")
            print(f"   Palabras: {text.split()}")
            print()

1. Archivo: fechas2/train/fechas2_000000.es.wav
   Texto: 'por favor el siguiente jueves'
   Palabras: ['por', 'favor', 'el', 'siguiente', 'jueves']

2. Archivo: fechas2/train/fechas2_000001.es.wav
   Texto: 'el viernes que viene'
   Palabras: ['el', 'viernes', 'que', 'viene']

3. Archivo: fechas2/train/fechas2_000002.es.wav
   Texto: 'el viernes gracias'
   Palabras: ['el', 'viernes', 'gracias']

4. Archivo: fechas2/train/fechas2_000003.es.wav
   Texto: 'el martes por favor'
   Palabras: ['el', 'martes', 'por', 'favor']

5. Archivo: fechas2/train/fechas2_000004.es.wav
   Texto: 'mañana'
   Palabras: ['mañana']

6. Archivo: fechas2/train/fechas2_000005.es.wav
   Texto: 'este viernes que viene gracias'
   Palabras: ['este', 'viernes', 'que', 'viene', 'gracias']

7. Archivo: fechas2/train/fechas2_000006.es.wav
   Texto: 'pasado mañana'
   Palabras: ['pasado', 'mañana']

8. Archivo: fechas2/train/fechas2_000007.es.wav
   Texto: 'por favor este martes que viene'
   Palabras: ['por', 'favor

In [9]:
class DigitSumTokenizer():    
    def __init__(self):
        # Vocabulario adaptado a la tarea de fechas2
        self.word2index = {
            'de': 0,
            'días': 1,
            'el': 2,
            'en': 3,
            'este': 4,
            'favor': 5,
            'gracias': 6,
            'jueves': 7,
            'lunes': 8,
            'martes': 9,
            'mañana': 10,
            'miércoles': 11,
            'par': 12,
            'pasado': 13,
            'por': 14,
            'próximo': 15,
            'que': 16,
            'siguiente': 17,
            'tres': 18,
            'un': 19,
            'viene': 20,
            'viernes': 21,
            '<pad>': 22,
            '<sos>': 23,
            '<eos>': 24,    
        }       
        self.index2word = {v:k for k,v in self.word2index.items()}
    
    def encode(self, x, seq_len=-1):
        x = '<sos> ' + x + ' <eos>'
        x = [self.word2index[w] for w in x.split()]
        if seq_len > len(x):
            x = x + [self.word2index['<pad>']] * (seq_len - len(x))
        return torch.tensor(x)

    def decode(self, x):
        if isinstance(x, torch.Tensor):
            x = x.tolist()
        x = ' '.join([self.index2word[i] for i in x])
        x = x.replace('<sos>', '').replace('<eos>', '').replace('<pad>', '')
        return x.strip()
    
tokenizer = DigitSumTokenizer()

class Digitsumset(torch.utils.data.Dataset):
    def __init__(self, file, seq_len=16):
        super().__init__()
        self.seq_len = seq_len
        with open(file, 'r', encoding='utf-8') as f:
            lines = f.readlines()[1:]  # Saltar la primera línea (encabezado)
            self.data = [line.strip().split(',', 1) for line in lines]  # split con maxsplit=1
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        wav_path, text = self.data[idx]
        return wav_path, text, tokenizer.encode(text, seq_len=self.seq_len)

trainset = Digitsumset('../fechas2/fechas2_train.es.csv')
testset = Digitsumset('../fechas2/fechas2_test.es.csv')

# Mostrar información del tokenizador
print(f"Tamaño del vocabulario: {len(tokenizer.word2index)}")
print(f"Palabras del vocabulario: {len(tokenizer.word2index) - 3} + 3 tokens especiales")

Tamaño del vocabulario: 25
Palabras del vocabulario: 22 + 3 tokens especiales


test the access to a sample of the dataset

In [10]:
x, y, yencoded = trainset[0]
print('input text:', x)
print('output text:', y)
print('output encoded:', yencoded, yencoded.shape)

input text: fechas2/train/fechas2_000000.es.wav
output text: por favor el siguiente jueves
output encoded: tensor([23, 14,  5,  2, 17,  7, 24, 22, 22, 22, 22, 22, 22, 22, 22, 22]) torch.Size([16])


# Train the network

In [11]:
model = Transformer(vocab_size=25,  # 22 palabras + 3 tokens especiales
                    eos_token_id=24,  # índice del token <eos>
                    nb_layers=4, 
                    d_model=128, d_ff=256, 
                    n_heads=8, d_head=16,
                    dropout=0.1, 
                    seq_len=16)  # longitud máxima observada + margen

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Usando dispositivo: {device}')
model.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

nb_epochs = 15
batch_size = 32
model.train()

trainloader = torch.utils.data.DataLoader(trainset, batch_size, shuffle=True)
for e in range(nb_epochs):
    avg_loss = 0
    for _,_,x in trainloader:
        x = x.to(device)
        opt.zero_grad()
        loss = model.loss(x)
        loss.backward()
        opt.step()
        avg_loss += loss.item()
    print('epoch %d/%d: avg_loss: %.2f' % (e,nb_epochs,avg_loss/len(trainloader)))
        
    
torch.save( [model, opt], 'model11.pt')

Causal Self Attention, d_model: 128 n_heads: 8 d_head: 16 seq_len: 16
Causal Self Attention, d_model: 128 n_heads: 8 d_head: 16 seq_len: 16
Causal Self Attention, d_model: 128 n_heads: 8 d_head: 16 seq_len: 16
Causal Self Attention, d_model: 128 n_heads: 8 d_head: 16 seq_len: 16


Usando dispositivo: cuda


epoch 0/15: avg_loss: 0.41


epoch 1/15: avg_loss: 0.34


epoch 2/15: avg_loss: 0.33


...

epoch 10/15: avg_loss: 0.33


epoch 11/15: avg_loss: 0.33


epoch 12/15: avg_loss: 0.33


epoch 13/15: avg_loss: 0.33


epoch 14/15: avg_loss: 0.33


# Test the network

In [12]:
#[model, opt] = torch.load('model11.pt')

# Ejemplos de prueba con frases de fechas
y = model.generate(tokenizer.encode('por favor el siguiente jueves')[:-1])
print(y)
print(tokenizer.decode(y))
print()

y = model.generate(tokenizer.encode('mañana')[:-1])
print(y)
print(tokenizer.decode(y))
print()

y = model.generate(tokenizer.encode('el próximo viernes gracias')[:-1])
print(y)
print(tokenizer.decode(y))

[23, 14, 5, 2, 17, 7, 24]
por favor el siguiente jueves

[23, 10, 24]
mañana

[23, 2, 15, 21, 6, 24]
el próximo viernes gracias


## Error rate in the test set

In [13]:
model.eval()
err = 0
for i, (wav_path, txt, _) in enumerate(testset):   
    y_pred = model.generate(tokenizer.encode(txt)[:-1])
    
    if txt != tokenizer.decode(y_pred):
        err += 1
print(f'error rate {err/len(testset):.2%},  ({err}/{len(testset)})')

error rate 0.00%,  (0/1000)
